In [8]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np

TO DO
- Railroads
- Utilities

In [9]:
# Property names kept separately for reference
# MISSING RR AND UTIL
property_names = [
    # 22 Streets
    "Mediterranean Avenue", "Baltic Avenue", "Oriental Avenue", "Vermont Avenue",
    "Connecticut Avenue", "St. Charles Place", "States Avenue", "Virginia Avenue",
    "St. James Place", "Tennessee Avenue", "New York Avenue", "Kentucky Avenue",
    "Indiana Avenue", "Illinois Avenue", "Atlantic Avenue", "Ventnor Avenue",
    "Marvin Gardens", "Pacific Avenue", "North Carolina Avenue", "Pennsylvania Avenue",
    "Park Place", "Boardwalk",
    
    # 4 Railroads
    "Reading Railroad", "Pennsylvania Railroad", "B. & O. Railroad", "Short Line Railroad",
    
    # 2 Utilities
    "Electric Company", "Water Works"
]

# Rent Matrix
# Columns: [Rent, Rent (1 House), Rent (2 Houses), Rent (3 Houses), Rent (4 Houses), Rent (Hotel)]
rent_arr = np.array([
    [2, 10, 30, 90, 160, 250],       # Mediterranean Avenue
    [4, 20, 60, 180, 320, 450],      # Baltic Avenue
    [6, 30, 90, 270, 400, 550],      # Oriental Avenue
    [6, 30, 90, 270, 400, 550],      # Vermont Avenue
    [8, 40, 100, 300, 450, 600],     # Connecticut Avenue
    [10, 50, 150, 450, 625, 750],    # St. Charles Place
    [10, 50, 150, 450, 625, 750],    # States Avenue
    [12, 60, 180, 500, 700, 900],    # Virginia Avenue
    [14, 70, 200, 550, 750, 950],    # St. James Place
    [14, 70, 200, 550, 750, 950],    # Tennessee Avenue
    [16, 80, 220, 600, 800, 1000],   # New York Avenue
    [18, 90, 250, 700, 875, 1050],   # Kentucky Avenue
    [18, 90, 250, 700, 875, 1050],   # Indiana Avenue
    [20, 100, 300, 750, 925, 1100],  # Illinois Avenue
    [22, 110, 330, 800, 975, 1150],  # Atlantic Avenue
    [22, 110, 330, 800, 975, 1150],  # Ventnor Avenue
    [24, 120, 360, 850, 1025, 1200], # Marvin Gardens
    [26, 130, 390, 900, 1100, 1275], # Pacific Avenue
    [26, 130, 390, 900, 1100, 1275], # North Carolina Avenue
    [28, 150, 450, 1000, 1200, 1400],# Pennsylvania Avenue
    [35, 175, 500, 1100, 1300, 1500],# Park Place
    [50, 200, 600, 1400, 1700, 2000], # Boardwalk
    
    [25, 50, 100, 200, 0, 0], # Reading RR (RR ONLY HAVE 4 OPTIONS)
    [25, 50, 100, 200, 0, 0], # Pennsylvania RR
    [25, 50, 100, 200, 0, 0], # B. & O. RR
    [25, 50, 100, 200, 0, 0], # Short Line RR
    
    [1, 0, 0, 0, 0, 0], # Electric Company (RENT BASED ON DICE)
    [1, 0, 0, 0, 0, 0], # Water Works
])

# binary purachase matrix (EXLUDES RRs and Utils)
purchased_arr = np.zeros((28, 5), dtype=int)

# Cost Matrix
# Columns: [Base Purchase, +1 House, +2 Houses, +3 Houses, +4 Houses, +Hotel]
cost_arr = np.array([
    [60,  110, 160, 210, 260, 310],  # Mediterranean Avenue
    [60,  110, 160, 210, 260, 310],  # Baltic Avenue
    [100, 150, 200, 250, 300, 350],  # Oriental Avenue
    [100, 150, 200, 250, 300, 350],  # Vermont Avenue
    [120, 170, 220, 270, 320, 370],  # Connecticut Avenue
    [140, 240, 340, 440, 540, 640],  # St. Charles Place
    [140, 240, 340, 440, 540, 640],  # States Avenue
    [160, 260, 360, 460, 560, 660],  # Virginia Avenue
    [180, 280, 380, 480, 580, 680],  # St. James Place
    [180, 280, 380, 480, 580, 680],  # Tennessee Avenue
    [200, 300, 400, 500, 600, 700],  # New York Avenue
    [220, 370, 520, 670, 820, 970],  # Kentucky Avenue
    [220, 370, 520, 670, 820, 970],  # Indiana Avenue
    [240, 390, 540, 690, 840, 990],  # Illinois Avenue
    [260, 410, 560, 710, 860, 1010], # Atlantic Avenue
    [260, 410, 560, 710, 860, 1010], # Ventnor Avenue
    [280, 430, 580, 730, 880, 1030], # Marvin Gardens
    [300, 500, 700, 900, 1100, 1300],# Pacific Avenue
    [300, 500, 700, 900, 1100, 1300],# North Carolina Avenue
    [320, 520, 720, 920, 1120, 1320],# Pennsylvania Avenue
    [350, 550, 750, 950, 1150, 1350],# Park Place
    [400, 600, 800, 1000, 1200, 1400],# Boardwalk
    
    [200, 200, 200, 200, 200, 200],  # Reading Railroad (cant be developed)
    [200, 200, 200, 200, 200, 200],  # Pennsylvania Railroad
    [200, 200, 200, 200, 200, 200],  # B. & O. Railroad
    [200, 200, 200, 200, 200, 200],   # Short Line Railroad

    [150, 0, 0, 0, 0, 0], # Electric Company (cant be developed)
    [150, 0, 0, 0, 0, 0] # Water Works
    
])

# Long Jail Probabilities
long_jail_props = [
    0.020073,   # Mediterranean Avenue
    0.020369,   # Baltic Avenue
    0.021317,   # Oriental Avenue
    0.021874,   # Vermont Avenue
    0.021680,   # Connecticut Avenue
    0.025560,   # St. Charles Place
    0.021741,   # States Avenue
    0.024255,   # Virginia Avenue
    0.026802,   # St. James Place
    0.028210,   # Tennessee Avenue
    0.028118,   # New York Avenue
    0.026143,   # Kentucky Avenue
    0.025671,   # Indiana Avenue
    0.029929,   # Illinois Avenue
    0.025370,   # Atlantic Avenue
    0.025191,   # Ventnor Avenue
    0.024381,   # Marvin Gardens
    0.025236,   # Pacific Avenue
    0.024721,   # North Carolina Avenue
    0.023531,   # Pennsylvania Avenue
    0.020595,   # Park Place
    0.024832,   # Boardwalk

    0.028010,   # Reading Railroad
    0.026354,   # Pennsylvania Railroad
    0.028930,   # B. & O. Railroad
    0.022906,   # Short Line Railroad

    0.026140,   # Electric Company
    0.026507    # Water Works
]

prob_arr_temp = np.array(long_jail_props)
prob_arr = prob_arr_temp.reshape(28, 1)

# Color Groups
color_groups = {
    "Brown": [0, 1],           # Mediterranean, Baltic
    "Light Blue": [2, 3, 4],   # Oriental, Vermont, Connecticut
    "Pink": [5, 6, 7],         # St. Charles, States, Virginia
    "Orange": [8, 9, 10],      # St. James, Tennessee, New York
    "Red": [11, 12, 13],       # Kentucky, Indiana, Illinois
    "Yellow": [14, 15, 16],    # Atlantic, Ventnor, Marvin Gardens
    "Green": [17, 18, 19],     # Pacific, North Carolina, Pennsylvania
    "Dark Blue": [20, 21]      # Park Place, Boardwalk
}


print(f"Cost Shape: {cost_arr.shape}")
print(f"Rent shape: {rent_arr.shape}")
print(f"Prob shape: {prob_arr.shape}")


expected_rent_matrix = prob_arr * rent_arr

Cost Shape: (28, 6)
Rent shape: (28, 6)
Prob shape: (28, 1)


In [10]:
m = gp.Model("model")
m.Params.OutputFlag = 0 

x = m.addMVar(shape=(28, 6), lb=0, vtype=GRB.BINARY, name="purchase")
# Y[group] = 1 if we own ALL properties in that group, 0 otherwise
Y = m.addVars(len(color_groups), vtype=GRB.BINARY, name="monopoly")

In [11]:
# CONSTRAINTS
# Budget
budget = 4500
m.addConstr(
    gp.quicksum(cost_arr[i, j] * x[i, j] for i in range(22) for j in range(6)) <= budget, 
    name="Budget"
)

# One level of development per property (Cant have one house and one hotel)
m.addConstr(x.sum(axis=1) <= 1, name="Max_One_Per_Property")

# --- 1. Monopoly and Build Gate Constraints ---
for g_idx, (color, prop_indices) in enumerate(color_groups.items()):
    num_props = len(prop_indices)

    # (1) You can only have the monopoly if you own EVERY property in the group
    # (Owned if Row sum = 1 )
    for i in prop_indices:
        m.addConstr(Y[g_idx] <= gp.quicksum(x[i, j] for j in range(6)), 
                    name=f"Monopoly_{color}_Req_{i}")

    # (2) If you own every property in the group, force the monopoly ON
    # Sum of all rows in the group - (num_props - 1) <= Y
    m.addConstr(
        gp.quicksum(x[i, j] for i in prop_indices for j in range(6)) - (num_props - 1) <= Y[g_idx],
        name=f"Monopoly_{color}_Force"
    )

    # (3) You can only build (j >= 1) if you have the monopoly
    for i in prop_indices:
        for j in range(1, 6):
            m.addConstr(x[i, j] <= Y[g_idx], name=f"Build_Gate_{i}_{j}")

# Dummy Var W will double BASE (j = 1) rent if monopoly is owned
# W[i] = 1 only if we own the base property (x[i,0] == 1) AND have the monopoly (Y[group] == 1)

W = m.addVars(22, vtype=GRB.BINARY, name="W_DoubleRent")
for g_idx, (color, prop_indices) in enumerate(color_groups.items()):
    for i in prop_indices:
        # W can only be 1 if we own the base property
        m.addConstr(W[i] <= x[i, 0], name=f"W_le_x_{i}")
        # W can only be 1 if we have the monopoly
        m.addConstr(W[i] <= Y[g_idx], name=f"W_le_Y_{i}")
        # Force W to be 1 if both conditions are true
        m.addConstr(W[i] >= x[i, 0] + Y[g_idx] - 1, name=f"W_force_{i}")


# Even development rule
# For each color group, compare every pair of properties
for color, prop_indices in color_groups.items():
    # If there's only 1 property, no need to check (though your groups have 2-3)
    if len(prop_indices) > 1:
        
        # Iterate through every pair of properties in the group
        for idx_a in range(len(prop_indices)):
            for idx_b in range(idx_a + 1, len(prop_indices)):
                
                prop_A = prop_indices[idx_a]
                prop_B = prop_indices[idx_b]
                
                # Calculate the actual development level (0 to 5) based on the binary choice
                # e.g., if x[A, 3] = 1, the level is 3.
                level_A = gp.quicksum(j * x[prop_A, j] for j in range(6))
                level_B = gp.quicksum(j * x[prop_B, j] for j in range(6))
                
                # Constraint: Level A cannot be more than 1 greater than Level B
                m.addConstr(
                    level_A - level_B <= 1, 
                    name=f"EvenBuild_{color}_{prop_A}_vs_{prop_B}_Max"
                )
                
                # Constraint: Level B cannot be more than 1 greater than Level A
                m.addConstr(
                    level_B - level_A <= 1, 
                    name=f"EvenBuild_{color}_{prop_A}_vs_{prop_B}_Min"
                )

# Railroads (22-25) and Utilities (26-27) cannot have houses/hotels
for i in range(22, 28):
    for j in range(1, 6):
        m.addConstr(x[i, j] == 0, name=f"No_Develop_{i}_{j}")

In [12]:
num_turns = 100
property_objective = (expected_rent_matrix * x).sum()

# Calculate the extra expected rent for doubled base properties
# prob_arr[i] is the probability, rent_arr[i, 0] is the base rent
extra_rent_expr = gp.quicksum(
    prob_arr[i, 0] * rent_arr[i, 0] * W[i] 
    for i in range(22)
)

objective = property_objective + extra_rent_expr

m.setObjective(num_turns * objective, GRB.MAXIMIZE)

In [13]:
m.optimize()
print(m.NumConstrs)

305


In [18]:
# --- 1. Create a reverse lookup from your existing color_groups ---
# This maps property index (0-21) -> Color Name string
prop_idx_to_color = {}
for color_name, indices in color_groups.items():
    for idx in indices:
        prop_idx_to_color[idx] = color_name

# --- 2. Readout Setup ---
print("\n" + "="*75)
print("MONOPOLY PURCHASE OPTIMIZATION RESULTS")
print("="*75)
print(f"Total Expected Revenue Score: {m.ObjVal:.4f}")
print("-" * 75)

total_spent = 0
purchases = []

solved_matrix = x.X 

for i in range(22):
    for j in range(6):
        if solved_matrix[i, j] > 0.5:
            cost = cost_arr[i, j]
            total_spent += cost
            
            if j == 0: level = "Base Property"
            elif j == 5: level = "Hotel"
            else: level = f"{j} House(s)"
            
            # Look up the color group
            group_name = prop_idx_to_color.get(i, "None")
            
            purchases.append(f"{property_names[i]:<25} | {group_name:<12} | {level:<15} | Cost: ${cost}")

# --- 3. Print Table ---
print(f"Total Money Spent: ${total_spent} / ${budget}")
print(f"Remaining Budget:  ${budget - total_spent}")
print("-" * 75)
print(f"{'Property':<25} | {'Color Group':<12} | {'Development':<15} | {'Cost'}")
print("-" * 75)
for p in purchases:
    print(p)
print("="*75)

print(f"NumConstrs: {m.NumConstrs}")   # total constraints


MONOPOLY PURCHASE OPTIMIZATION RESULTS
Total Expected Revenue Score: 15326.3047
---------------------------------------------------------------------------
Total Money Spent: $4470 / $4500
Remaining Budget:  $30
---------------------------------------------------------------------------
Property                  | Color Group  | Development     | Cost
---------------------------------------------------------------------------
Oriental Avenue           | Light Blue   | Hotel           | Cost: $350
Vermont Avenue            | Light Blue   | Hotel           | Cost: $350
Connecticut Avenue        | Light Blue   | Hotel           | Cost: $370
St. Charles Place         | Pink         | 3 House(s)      | Cost: $440
States Avenue             | Pink         | 3 House(s)      | Cost: $440
Virginia Avenue           | Pink         | 3 House(s)      | Cost: $460
St. James Place           | Orange       | Hotel           | Cost: $680
Tennessee Avenue          | Orange       | Hotel           | Cost